# Preprocessing playground

Try preprocessing ideas on the H-alpha images and see what each one brings out.

- **Section 1** holds every setting you are likely to change. Edit it, then re-run the cells below.
- **Sections 3-6** are the individual techniques (limb-darkening correction, CLAHE, background subtraction, ridge filters).
- **Section 7** compares them all side by side; **section 8** has sliders.

Images are 2048x2048. Use `CROP` to zoom into a region at full resolution, which is where thin barbs and fibrils are visible.

## 1. Settings (edit these)

In [ ]:
FILE = "20110109104734Ch.jpeg"     # any file in data/raw/train/train_images
SHOW_ANNOTATIONS = True            # outline the ground-truth filaments
ANNOTATOR = 0                      # which annotator's version to draw (0, 1, 2 ...; wraps around if fewer)
CROP = None                        # None = whole image, or (x, y, size) in full-res pixels, e.g. (700, 250, 600)

# Limb-darkening correction
LIMB_SMOOTH = 8                    # smoothing of the radial brightness profile (larger = smoother)

# CLAHE (local contrast)
CLAHE_CLIP = 0.02                  # contrast limit; higher = stronger
CLAHE_TILE = 128                   # tile size in pixels; smaller = more local

# Background subtraction
BG_SIGMA = 40                      # blur size of the "background"; smaller removes larger-scale structure too

# Ridge filters (Frangi), for thin dark lines
FIBRIL_SIGMAS = (1, 1.5, 2)        # small scales: fibrils / mottles / thin barbs
FILAMENT_SIGMAS = (3, 4, 6)        # larger scales: whole filament bodies

: 

## 2. Setup

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from scipy import ndimage as ndi
from skimage import exposure, filters, measure

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
IMAGE_DIR = ROOT / "data/raw/train/train_images"
ANN_FILE = ROOT / "data/raw/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json"

coco = json.load(open(ANN_FILE))
entries = {}
for im in coco["images"]:
    entries.setdefault(im["file_name"], []).append(im["id"])
anns_by_image = {}
for a in coco["annotations"]:
    anns_by_image.setdefault(a["image_id"], []).append(a)


def load_gray(file):
    return np.asarray(Image.open(IMAGE_DIR / file).convert("L"), dtype=np.float32) / 255.0


def get_annotations(file, annotator=0):
    ids = sorted(entries[file])
    return anns_by_image.get(ids[annotator % len(ids)], [])


def find_disc(img):
    """Estimate the solar disc centre and radius (works on a 4x downsample)."""
    small = ndi.gaussian_filter(img[::4, ::4], 2)
    thr = filters.threshold_multiotsu(small, classes=3)[1]
    labels = measure.label(small > thr)
    biggest = max(measure.regionprops(labels), key=lambda r: r.area)
    cy, cx = biggest.centroid
    radius = np.sqrt(biggest.area / np.pi)
    return cy * 4, cx * 4, radius * 4


def normalize(x, mask=None, lo=1, hi=99):
    """Percentile stretch to 0..1 for display, using only pixels inside `mask` if given."""
    vals = x[mask] if mask is not None else x
    a, b = np.percentile(vals, [lo, hi])
    return np.clip((x - a) / max(b - a, 1e-8), 0, 1)


def show(panels, titles=None, crop=None, annotations=None, cols=3, size=6):
    """Show several images in a grid, optionally cropped and with annotation outlines."""
    rows = int(np.ceil(len(panels) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(size * cols, size * rows), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    x0, y0, s = crop if crop else (0, 0, panels[0].shape[1])
    for i, p in enumerate(panels):
        ax = axes.ravel()[i]
        ax.imshow(p[y0:y0 + s, x0:x0 + s], cmap="gray", extent=(x0, x0 + s, y0 + s, y0), vmin=0, vmax=1)
        if annotations:
            for a in annotations:
                poly = np.array(a["segmentation"][0]).reshape(-1, 2)
                ax.plot(poly[:, 0], poly[:, 1], color="cyan", lw=0.8)
            ax.set_xlim(x0, x0 + s)
            ax.set_ylim(y0 + s, y0)
        if titles:
            ax.set_title(titles[i])
    plt.tight_layout()
    plt.show()

## 3. Load the image and find the disc

In [ ]:
img = load_gray(FILE)
cy, cx, radius = find_disc(img)
yy, xx = np.indices(img.shape)
dist = np.hypot(yy - cy, xx - cx)
disc = dist < radius * 0.985          # slightly inside the edge to avoid limb artefacts
print(f"{FILE}: disc centre=({cx:.0f}, {cy:.0f}) radius={radius:.0f}px")

anns = get_annotations(FILE, ANNOTATOR) if SHOW_ANNOTATIONS else None
show([normalize(img), normalize(img, disc)], ["raw", "raw, contrast-stretched on the disc"],
     crop=CROP, annotations=anns, cols=2, size=8)

## 4. Limb-darkening correction

The disc is darker towards the edge. We measure the average brightness at each distance from the centre and divide it out,
so the same filament looks the same anywhere on the disc.

In [ ]:
def limb_correct(img, dist, radius, smooth=LIMB_SMOOTH):
    inside = dist < radius
    bins = dist[inside].astype(int)
    profile = np.bincount(bins, weights=img[inside]) / np.maximum(np.bincount(bins), 1)
    profile = ndi.gaussian_filter1d(profile, smooth, mode="nearest")
    flat = np.where(inside, img / np.maximum(profile[np.minimum(dist.astype(int), len(profile) - 1)], 1e-3), 0)
    return flat * img[inside].mean() / max(flat[inside].mean(), 1e-8)


flat = limb_correct(img, dist, radius)
flat_n = normalize(flat, disc) * disc
show([normalize(img, disc) * disc, flat_n], ["raw (stretched)", "limb-darkening corrected"],
     crop=CROP, annotations=anns, cols=2, size=8)

## 5. CLAHE and background subtraction

- **CLAHE** boosts contrast locally. This is the likely "enhanced-contrast" input for the second model.
- **Background subtraction** removes large, smooth brightness variation and keeps small structures.

In [ ]:
clahe = exposure.equalize_adapthist(flat_n, kernel_size=CLAHE_TILE, clip_limit=CLAHE_CLIP) * disc

background = ndi.gaussian_filter(flat, BG_SIGMA)
bgsub = normalize((flat - background) * disc, disc) * disc

show([flat_n, clahe, bgsub], ["limb-corrected", f"CLAHE (clip={CLAHE_CLIP}, tile={CLAHE_TILE})",
                              f"background subtracted (sigma={BG_SIGMA})"],
     crop=CROP, annotations=anns, cols=3, size=7)

## 6. Ridge filters: fibrils vs filaments

A Frangi filter responds to thin dark lines. Small scales light up the short streaks over the disc (fibrils and mottles)
and thin barbs; larger scales light up the bodies of real filaments.

In [ ]:
def ridge(img_flat, sigmas, mask):
    r = filters.frangi(img_flat, sigmas=sigmas, black_ridges=True)
    r = r * ndi.binary_erosion(mask, iterations=int(max(sigmas) * 3 + 5))   # drop the limb response
    return normalize(r, mask, 50, 99.7) * mask


ridge_small = ridge(flat_n, FIBRIL_SIGMAS, disc)
ridge_large = ridge(flat_n, FILAMENT_SIGMAS, disc)
show([flat_n, ridge_small, ridge_large],
     ["limb-corrected", f"ridge, small scale {FIBRIL_SIGMAS}", f"ridge, large scale {FILAMENT_SIGMAS}"],
     crop=CROP, annotations=anns, cols=3, size=7)

## 7. Everything side by side

In [ ]:
show([normalize(img, disc) * disc, flat_n, clahe, bgsub, ridge_small, ridge_large],
     ["raw", "limb-corrected", "CLAHE", "background subtracted", "ridge (small)", "ridge (large)"],
     crop=CROP, annotations=anns, cols=3, size=7)

## 8. Interactive sliders

Pick a technique and drag the sliders. Use the crop controls to zoom in. Needs `ipywidgets`; if the sliders do not show up,
use the settings in section 1 instead.

In [ ]:
try:
    from ipywidgets import Dropdown, FloatSlider, IntSlider, interact

    def explore(method="CLAHE", clip=0.02, tile=128, bg_sigma=40, ridge_sigma=2.0, x=700, y=250, size=600,
                annotate=True):
        if method == "CLAHE":
            out = exposure.equalize_adapthist(flat_n, kernel_size=int(tile), clip_limit=clip) * disc
        elif method == "background subtracted":
            out = normalize((flat - ndi.gaussian_filter(flat, bg_sigma)) * disc, disc) * disc
        elif method == "ridge":
            out = ridge(flat_n, (ridge_sigma,), disc)
        else:
            out = flat_n
        show([flat_n, out], ["limb-corrected", method], crop=(x, y, size),
             annotations=get_annotations(FILE, ANNOTATOR) if annotate else None, cols=2, size=8)

    interact(explore,
             method=Dropdown(options=["limb-corrected", "CLAHE", "background subtracted", "ridge"], value="CLAHE"),
             clip=FloatSlider(min=0.005, max=0.1, step=0.005, value=0.02),
             tile=IntSlider(min=32, max=512, step=32, value=128),
             bg_sigma=IntSlider(min=5, max=100, step=5, value=40),
             ridge_sigma=FloatSlider(min=0.5, max=8, step=0.5, value=2.0),
             x=IntSlider(min=0, max=1800, step=50, value=700),
             y=IntSlider(min=0, max=1800, step=50, value=250),
             size=IntSlider(min=200, max=2048, step=100, value=600),
             annotate=True)
except ImportError:
    print("ipywidgets is not installed; use the settings in section 1.")

## 9. Dense optical flow (mind the limits of this dataset)

Dense optical flow estimates a motion vector for every pixel between two frames of the same scene.

**This dataset is a poor fit.** The median gap between consecutive images is about 2.8 days, and only 9 pairs are within
an hour of each other. Those closest pairs are 20 seconds apart but come from *different stations* (the letter before
`h.jpeg`: L vs U, C vs B, ...), so the flow mostly measures differences between instruments (shift, rotation, scale),
not motion on the sun. The test set is also single images, so flow cannot be a model input at prediction time.

The cell below still runs it on the closest pair so you can see what comes out. It is a useful way to check how well two stations
line up; do not read the arrows as solar flow. Section 10 has a single-image alternative.

In [ ]:
from skimage.registration import optical_flow_ilk

FLOW_FILE_A = "20141013195834Ch.jpeg"   # try other pairs: 20150625082014Th.jpeg / 20150625082034Lh.jpeg
FLOW_FILE_B = "20141013195854Bh.jpeg"
FLOW_DOWNSAMPLE = 4                     # compute on a smaller image (faster, smoother)
FLOW_RADIUS = 15                        # window size; larger = smoother, less detailed field
FLOW_ARROW_STEP = 24                    # one arrow every N low-res pixels


def flow_between(a, b, down=FLOW_DOWNSAMPLE, radius=FLOW_RADIUS):
    a_s = ndi.gaussian_filter(a, down / 2)[::down, ::down]
    b_s = ndi.gaussian_filter(b, down / 2)[::down, ::down]
    v, u = optical_flow_ilk(a_s, b_s, radius=radius)     # row and column displacement
    return a_s, b_s, v * down, u * down                   # displacements in full-res pixels


img_a, img_b = load_gray(FLOW_FILE_A), load_gray(FLOW_FILE_B)
a_s, b_s, v, u = flow_between(img_a, img_b)
mag = np.hypot(u, v)
print(f"median displacement {np.median(mag):.1f}px, mean shift (dx={u.mean():.1f}, dy={v.mean():.1f}) full-res pixels")

fig, axes = plt.subplots(1, 3, figsize=(21, 7))
axes[0].imshow(a_s, cmap="gray"); axes[0].set_title(f"A: {FLOW_FILE_A}")
axes[1].imshow(b_s, cmap="gray"); axes[1].set_title(f"B: {FLOW_FILE_B}")
axes[2].imshow(a_s, cmap="gray")
im = axes[2].imshow(mag, cmap="magma", alpha=0.5)
ys, xs = np.mgrid[0:mag.shape[0]:FLOW_ARROW_STEP, 0:mag.shape[1]:FLOW_ARROW_STEP]
axes[2].quiver(xs, ys, u[ys, xs], -v[ys, xs], color="cyan", angles="uv", scale_units="xy", scale=FLOW_DOWNSAMPLE * 0.5)
axes[2].set_title("flow magnitude (px) + arrows")
plt.colorbar(im, ax=axes[2], fraction=0.046)
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 10. Streak orientation (single-image alternative)

The short streaks over the disc follow the local magnetic field. A **structure tensor** measures, for every pixel, the
dominant direction of the texture and how coherent it is (1 = all streaks parallel, 0 = no preferred direction).
The sticks below show the streak direction where the coherence is high enough.

In [ ]:
from skimage.feature import structure_tensor

ORIENT_CROP = (700, 250, 600)     # (x, y, size); zoom in, the sticks are dense
ORIENT_SMOOTH = 1.5               # smooths the image before taking gradients (streak thickness)
ORIENT_WINDOW = 6                 # region over which direction is averaged (larger = smoother field)
ORIENT_MIN_COHERENCE = 0.4        # only draw sticks where the streaks are clearly aligned
ORIENT_STEP = 12                  # one stick every N pixels

smoothed = ndi.gaussian_filter(flat_n, ORIENT_SMOOTH)
Arr, Arc, Acc = structure_tensor(smoothed, sigma=ORIENT_WINDOW, order="rc")
theta = 0.5 * np.arctan2(2 * Arc, Acc - Arr) + np.pi / 2       # streak direction (perpendicular to gradient)
coherence = np.sqrt((Acc - Arr) ** 2 + 4 * Arc ** 2) / np.maximum(Acc + Arr, 1e-12)
coherence *= ndi.binary_erosion(disc, iterations=ORIENT_WINDOW * 3)

x0, y0, s = ORIENT_CROP
ys, xs = np.mgrid[y0:y0 + s:ORIENT_STEP, x0:x0 + s:ORIENT_STEP]
keep = coherence[ys, xs] > ORIENT_MIN_COHERENCE

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
axes[0].imshow(coherence[y0:y0 + s, x0:x0 + s], cmap="viridis", extent=(x0, x0 + s, y0 + s, y0), vmin=0, vmax=1)
axes[0].set_title("coherence (how aligned the streaks are)")
axes[1].imshow(flat_n[y0:y0 + s, x0:x0 + s], cmap="gray", extent=(x0, x0 + s, y0 + s, y0), vmin=0, vmax=1)
axes[1].quiver(xs[keep], ys[keep], np.cos(theta[ys, xs])[keep] * 1.0, -np.sin(theta[ys, xs])[keep] * 1.0,
               color="cyan", angles="uv", pivot="mid", headwidth=0, headlength=0, headaxislength=0, width=0.003, scale=45)
if anns:
    for a in anns:
        poly = np.array(a["segmentation"][0]).reshape(-1, 2)
        axes[1].plot(poly[:, 0], poly[:, 1], color="yellow", lw=0.8)
    axes[1].set_xlim(x0, x0 + s); axes[1].set_ylim(y0 + s, y0)
axes[1].set_title("streak direction (cyan) and filament outlines (yellow)")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 11. Do the streaks line up with filaments? (measurement)

Filaments sit in "filament channels", and the small streaks next to them may run parallel to the filament's axis.
This cell measures that on many training images: for each clearly elongated filament it takes the streak direction in a ring
around it (from the structure tensor in section 10) and scores how parallel it is to the filament's own axis.

- **1** = parallel, **0** = random, **-1** = perpendicular.
- "far" is the same score measured far from every filament, so it is the no-effect baseline (about 0).
- Rings that start right at the filament edge are inflated, because the filament's own edges are aligned with its axis.
  Look at the rings further out for the real effect.

Running it takes about a minute for 40 images.

In [ ]:
from PIL import ImageDraw
from skimage.feature import structure_tensor

ALIGN_N_IMAGES = 40                            # how many random training images to measure
ALIGN_SEED = 1                                 # change to sample different images
ALIGN_RING_BANDS = [(4, 30), (20, 50), (35, 70)]   # distance from the filament edge in pixels (inner, outer)
ALIGN_FAR_DISTANCE = 90                        # "far" region = at least this many pixels from any filament
ALIGN_MIN_AREA = 600                           # skip small filaments (pixels)
ALIGN_MIN_ELONGATION = 4                       # skip round filaments (ratio of principal axis variances)


def streak_orientation(img_flat):
    """Per-pixel streak direction (radians, image x/y coordinates) and coherence (0..1)."""
    sm = ndi.gaussian_filter(img_flat, ORIENT_SMOOTH)
    Arr, Arc, Acc = structure_tensor(sm, sigma=ORIENT_WINDOW, order="rc")
    theta = 0.5 * np.arctan2(2 * Arc, Acc - Arr) + np.pi / 2
    coh = np.sqrt((Acc - Arr) ** 2 + 4 * Arc ** 2) / np.maximum(Acc + Arr, 1e-12)
    return theta, coh


def polygon_mask(ann, shape):
    m = Image.new("L", (shape[1], shape[0]))
    ImageDraw.Draw(m).polygon([float(v) for v in ann["segmentation"][0]], fill=1)
    return np.array(m, bool)


def measure_alignment(file):
    """Returns {band: [(ring_alignment, far_alignment), ...]} with one entry per elongated filament."""
    image = load_gray(file)
    cy_, cx_, r_ = find_disc(image)
    yy_, xx_ = np.indices(image.shape)
    d_ = np.hypot(yy_ - cy_, xx_ - cx_)
    disc_ = d_ < r_ * 0.985
    flat_ = normalize(limb_correct(image, d_, r_), disc_) * disc_
    theta_, coh_ = streak_orientation(flat_)
    inner = ndi.binary_erosion(disc_, iterations=25)

    masks = [polygon_mask(a, image.shape) for a in get_annotations(file, ANNOTATOR)]
    if not masks:
        return {}
    all_masks = np.any(masks, axis=0)
    far = inner & ~ndi.binary_dilation(all_masks, iterations=ALIGN_FAR_DISTANCE)

    out = {b: [] for b in ALIGN_RING_BANDS}
    for m in masks:
        if m.sum() < ALIGN_MIN_AREA:
            continue
        ys_, xs_ = np.nonzero(m)
        w, v = np.linalg.eigh(np.cov(np.vstack([xs_, ys_])))
        if w[1] / max(w[0], 1e-6) < ALIGN_MIN_ELONGATION:
            continue
        phi = np.arctan2(v[1, 1], v[0, 1])                      # filament axis direction

        def score(region):
            return float(np.average(np.cos(2 * (theta_[region] - phi)), weights=coh_[region]))

        for lo, hi in ALIGN_RING_BANDS:
            ring = ndi.binary_dilation(m, iterations=hi) & ~ndi.binary_dilation(m, iterations=lo) & inner & ~all_masks
            if ring.sum() >= 200:
                out[(lo, hi)].append((score(ring), score(far)))
    return out


import random
sample = sorted(entries)
random.Random(ALIGN_SEED).shuffle(sample)
results = {b: [] for b in ALIGN_RING_BANDS}
for f in sample[:ALIGN_N_IMAGES]:
    for b, vals in measure_alignment(f).items():
        results[b] += vals

fig, ax = plt.subplots(figsize=(9, 5))
labels, data = [], []
for b, vals in results.items():
    arr = np.array(vals)
    labels += [f"ring {b[0]}-{b[1]}px"]
    data += [arr[:, 0]]
    print(f"ring {b[0]}-{b[1]}px: n={len(arr)}  mean {arr[:, 0].mean():.3f}  median {np.median(arr[:, 0]):.3f}  "
          f"far-baseline {arr[:, 1].mean():.3f}  more parallel than far: {np.mean(arr[:, 0] > arr[:, 1]):.0%}")
labels += ["far (baseline)"]
data += [np.array(results[ALIGN_RING_BANDS[0]])[:, 1]]
ax.boxplot(data, labels=labels)
ax.axhline(0, color="gray", lw=0.8)
ax.set_ylabel("alignment with filament axis  (1 = parallel, 0 = random)")
plt.tight_layout()
plt.show()

## Ideas to try next

- Change `FILE` to an image from a different observatory (the letter before `h.jpeg` in the file name: B, C, L, M, T, U) and check the disc detection and contrast still look right.
- Stack `flat_n`, `clahe` and `ridge_small` as the three input channels of the model instead of copying grayscale three times.
- Judge each technique by whether the cyan filament outlines stand out more clearly against the background, especially for thin barbs and small filaments.